# Phase 7 - Step 1: Prior Case Retrieval (PCR) data exploration

**Task:** for every _query case_, rank the cases of the same split's _candidate pool_ and submit the **top 7**
(exactly 7 unique ids, best first). **Metric: Micro-F1@7** - count the hits (predicted ids that are in the gold set)
over _all_ queries together, then precision = hits / (7 x queries), recall = hits / (all gold ids), F1 = their
harmonic mean. The gold set of a query can have fewer or more than 7 ids.

What this notebook finds out before we design any model (CPU only, no training):

1. download `Phase_7_PCR/` from Hugging Face (skipped if already on disk)
2. the real file format (list or dict, ids, sentences, label field)
3. data checks: duplicates, empty texts, is a query inside its own pool, overlap between the splits, placeholders
4. the labels: how many relevant cases per query, the **best possible Micro-F1@7**, popular candidates
5. lengths in words and **bpe32k tokens** -> how much of a case my Phase-1 models (context 1536) can read
6. **years**: are the relevant cases really older than the query ("prior")? can a year filter remove candidates?
7. **BM25 baselines** (a classical word-matching ranker, no learning) on dev: settings, which part of the case
   matters (citation sentences, first N words), recall@K (how deep a re-ranker must look), failure example
8. a checked reference `submission.zip` from BM25 (**not for upload**: it does not use a Phase-1 model)

Results are cached in `experiments/phase7/p7_01_data/` (re-running loads them).


In [1]:
import json, re, sys, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# project root = first folder upwards that contains .gitignore (same as all earlier notebooks)
ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import pcr_utils as pu

RAW_DIR = ROOT / "data" / "raw"
P7 = pu.pcr_dir(ROOT)
OUT = ROOT / "experiments" / "phase7" / "p7_01_data"
OUT.mkdir(parents=True, exist_ok=True)
TOK_PATH = ROOT / "data" / "tokenizer" / "bpe32k.json"
N_TOK_SAMPLE = 1500       # documents per split / role used for the bpe32k token statistics
FWD_TOK_PER_S = 50_000    # ROUGH forward-only speed of a 70M Phase-1 model on the RTX 4050 (for time estimates only)
K = pu.TOP_K
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("project root:", ROOT)

def cached(name, fn):
    # load OUT/name if it exists, else compute fn(), save and return it
    p = OUT / name
    if p.exists():
        print(f"[cache] {name}")
        return json.loads(p.read_text(encoding="utf-8"))
    obj = fn()
    p.write_text(json.dumps(obj, indent=1), encoding="utf-8")
    return obj

def pct_row(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return {}
    return {"n": len(x), "mean": round(float(x.mean()), 1), "p10": int(np.percentile(x, 10)), "p50": int(np.percentile(x, 50)),
            "p90": int(np.percentile(x, 90)), "p99": int(np.percentile(x, 99)), "max": int(x.max())}

project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download the Phase 7 files (skipped if already on disk)

The doc (section 13.3): `Train/train.json` + `Train/train_candidates.json`, the same for `Dev/` and `Test/`.
The repository is gated: you must be logged in (`hf auth login`, done in earlier phases). ~330 MB in total.


In [2]:
NEEDED = {f"{s}{x}" for s in pu.SPLITS for x in ("", "_candidates")}
FILES = pu.find_files(ROOT)
if NEEDED <= set(FILES):
    print("Phase 7 data already downloaded -> skipping download")
else:
    from huggingface_hub import snapshot_download
    snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                      allow_patterns="Phase_7_PCR/*", local_dir=str(RAW_DIR))
    FILES = pu.find_files(ROOT)
for p in sorted(P7.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(P7)):40s} {p.stat().st_size / 1e6:9.2f} MB")
missing = NEEDED - set(FILES)
assert not missing, f"missing files: {missing}"

Phase 7 data already downloaded -> skipping download
Dev\dev.json                                  5.87 MB
Dev\dev_candidates.json                      51.89 MB
Test\test.json                                9.34 MB
Test\test_candidates.json                    71.14 MB
Train\train.json                             33.19 MB
Train\train_candidates.json                 159.78 MB


## 2. Load the files and look at the real format

We do not trust the doc's description blindly (Phase 4's test file secretly had labels, Phase 6 had empty
summaries). `pu.load_pcr` accepts a list of records or a dict `{id: record}`, finds the id / text / label keys,
keeps **ids as strings** (`"0000021652"`: turning it into the number 21652 would lose the leading zeros and the
evaluator would not recognise it), and drops test labels if any exist.


In [3]:
if "FILES" not in globals():          # the download cell was skipped (e.g. after a kernel restart)
    FILES = pu.find_files(ROOT)
t0 = time.time()
DATA, INFO = pu.load_pcr(files=FILES)
print(f"loaded in {time.time() - t0:.0f} s")
rows = []
for split, rec in INFO.items():
    for role in ("queries", "pool"):
        r = rec[role]
        rows.append({"split/role": f"{split}/{role}", "file": r["file"], "json": r["file_type"], "records": r["records"],
                     "keys": r["keys"], "id type": r["id_type_in_file"], "text type": r["text_type"], "label key": r["rel_key"]})
display(pd.DataFrame(rows).set_index("split/role"))
for split, rec in INFO.items():
    extra = {k: v for k, v in rec.items() if k not in ("queries", "pool")}
    print(split, extra)
    if rec.get("test_had_labels"):
        print("WARNING: test.json contained labels -> dropped now, never used")

q0, c0 = DATA["train"]["queries"][0], DATA["train"]["pool"][0]
print(f"\nfirst train query id={q0['id']!r}: {len(q0['sents'])} sentences, {len(q0['rel'])} relevant: {q0['rel'][:10]}")
for s in q0["sents"][:4]:
    print("   |", s[:250])
print(f"first train candidate id={c0['id']!r}: {len(c0['sents'])} sentences")
for s in c0["sents"][:4]:
    print("   |", s[:250])

loaded in 2 s


,file,json,records,keys,id type,text type,label key
split/role,,,,,,,
train/queries,train.json,dict,827,"[text, relevant_candidates, id]",str,list,relevant_candidates
train/pool,train_candidates.json,dict,4320,"[text, id]",str,list,NaN
dev/queries,dev.json,dict,118,"[text, relevant_candidates, id]",str,list,relevant_candidates
dev/pool,dev_candidates.json,dict,1023,"[text, id]",str,list,NaN
test/queries,test.json,dict,237,"[text, id]",str,list,NaN
test/pool,test_candidates.json,dict,1727,"[text, id]",str,list,NaN


train {'gold_missing_from_pool_share': 0.0}
dev {'gold_missing_from_pool_share': 0.0}
test {}

first train query id='0000000599': 54 sentences, 2 relevant: ['0001292810', '0001742283']
   | PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 15
   | CIVIL APPELLATE JURISDICTION: Civil Appeal Nos. 30 and 31 of 1963. 
   | Appeals by special leave from the judgment and order dated August 31, 1962, of <ORG> in D. B Civil Writ Petitions Nos. 376 and 377 of 1962.' <NAME>, <NAME>, <NAME>, <NAME> and <NAME> for the appellant (in C.A. No. 30 of 1963). 
   | <NAME>, <NAME>, <NAME> and <NAME>, for the appellant (in C.A. No 31 of 1963). 
first train candidate id='0000000262': 55 sentences
   | JUDGMENT <NAME>, C.J. 
   | 1. The only point which has to be considered in this appeal is whether or not the lo

## 3. Data checks

Per split: duplicate ids, empty cases, queries without gold, gold ids that are not in the pool, **is the query
itself inside its own pool** (then a text ranker would put it first - we must exclude it), identical texts,
line breaks / non-ASCII (my Phase-1 tokenizer was trained on cleaned ASCII text), placeholders like `<CITATION>`
(IL-PCR-style data hides the citation itself, otherwise the task would be trivial), and overlap between the splits.


In [4]:
NONASCII = re.compile(r"[^\x00-\x7f]")

def check_split(split):
    Q, C = DATA[split]["queries"], DATA[split]["pool"]
    pool_ids = {d["id"] for d in C}
    qtexts = {d["id"]: pu.text_key(pu.doc_text(d)) for d in Q}
    ctexts = collections.Counter(pu.text_key(pu.doc_text(d)) for d in C)
    ctext_set = set(ctexts)
    gold = [c for q in Q for c in (q["rel"] or [])]
    allt = [pu.doc_text(d) for d in Q + C]
    nchar = sum(map(len, allt))
    return {
        "queries": len(Q), "pool": len(C),
        "dup query ids": len(Q) - len({d["id"] for d in Q}), "dup pool ids": len(C) - len(pool_ids),
        "empty queries": sum(not pu.doc_text(d).strip() for d in Q), "empty candidates": sum(not pu.doc_text(d).strip() for d in C),
        "queries w/o gold": sum(1 for q in Q if q["rel"] is not None and not q["rel"]) if gold else "-",
        "gold ids not in pool": sum(c not in pool_ids for c in gold) if gold else "-",
        "query id in own pool": sum(d["id"] in pool_ids for d in Q),
        "query = own gold": sum(q["id"] in (q["rel"] or []) for q in Q),
        "query text in pool": sum(h in ctext_set for h in qtexts.values()),
        "dup texts in pool (extra copies)": sum(v - 1 for v in ctexts.values() if v > 1),
        "line breaks %": round(100 * np.mean(["\n" in t for t in allt]), 1),
        "non-ASCII % chars": round(100 * sum(len(NONASCII.findall(t)) for t in allt) / max(nchar, 1), 3),
        "upper-case letters %": round(100 * sum(sum(ch.isupper() for ch in t[:2000]) for t in allt) / max(sum(len(t[:2000]) for t in allt), 1), 1),
    }

CHECKS = {s: check_split(s) for s in tqdm(DATA, desc="checks")}
display(pd.DataFrame(CHECKS))

# placeholders (e.g. <CITATION>): which ones, in how many cases, how many per case
ph_rows, PH = [], collections.Counter()
for split in DATA:
    for role in ("queries", "pool"):
        docs = DATA[split][role]
        c = collections.Counter(m for d in docs for s in d["sents"] for m in pu.PLACEHOLDER_RE.findall(s))
        PH.update(c)
        ph_rows.append({"split/role": f"{split}/{role}", "cases with any %": round(100 * np.mean([any(pu.PLACEHOLDER_RE.search(s) for s in d["sents"]) for d in docs]), 1),
                        "per case": round(sum(c.values()) / max(len(docs), 1), 2), "top": c.most_common(4)})
display(pd.DataFrame(ph_rows).set_index("split/role"))
# <NAME> / <ORG> are masked names; <CITATION> is the masked citation = the marker we need
CIT = "<CITATION>" if "<CITATION>" in PH else (PH.most_common(1)[0][0] if PH else None)
print("citation marker used below:", CIT)

# Supreme-Court-style header ("PETITIONER: ... DATE OF JUDGMENT: ... CITATION: ... CITATOR INFO: ...")
hdr = []
for split in DATA:
    for role in ("queries", "pool"):
        docs = DATA[split][role]
        heads = [pu.doc_text(d)[:6000] for d in docs]
        hdr.append({"split/role": f"{split}/{role}", "DATE OF JUDGMENT header %": round(100 * np.mean(["DATE OF JUDGMENT" in h for h in heads]), 1),
                    "CITATOR INFO %": round(100 * np.mean(["CITATOR INFO" in h for h in heads]), 1)})
display(pd.DataFrame(hdr).set_index("split/role"))

# overlap between splits (ids and texts)
ov = []
splits = list(DATA)
for i, a in enumerate(splits):
    for b in splits[i + 1:]:
        pa, pb = {d["id"] for d in DATA[a]["pool"]}, {d["id"] for d in DATA[b]["pool"]}
        qa, qb = {d["id"] for d in DATA[a]["queries"]}, {d["id"] for d in DATA[b]["queries"]}
        ta = {pu.text_key(pu.doc_text(d)) for d in DATA[a]["pool"]}
        tb = {pu.text_key(pu.doc_text(d)) for d in DATA[b]["pool"]}
        ov.append({"pair": f"{a} vs {b}", "pool ids shared": len(pa & pb), "pool texts shared": len(ta & tb),
                   "queries shared": len(qa & qb), f"{a} queries in {b} pool": len(qa & pb), f"{b} queries in {a} pool": len(qb & pa)})
display(pd.DataFrame(ov).set_index("pair").fillna(0).astype(int))

checks: 100%|██████████| 3/3 [00:15<00:00,  5.00s/it]


,train,dev,test
queries,827.000,118.000,237
pool,4320.000,1023.000,1727
dup query ids,0.000,0.000,0
dup pool ids,0.000,0.000,0
empty queries,0.000,0.000,0
empty candidates,0.000,0.000,0
queries w/o gold,10.000,0.000,-
gold ids not in pool,0.000,0.000,-
query id in own pool,827.000,118.000,237
query = own gold,0.000,0.000,0


,cases with any %,per case,top
split/role,,,
train/queries,100.0,152.09,"[(<ORG>, 74366), (<NAME>, 43415), (<CITATION>,..."
train/pool,99.8,142.98,"[(<ORG>, 371599), (<NAME>, 212895), (<CITATION..."
dev/queries,100.0,201.36,"[(<ORG>, 15929), (<NAME>, 6328), (<CITATION>, ..."
dev/pool,99.9,203.26,"[(<ORG>, 136464), (<NAME>, 61107), (<CITATION>..."
test/queries,100.0,146.40,"[(<ORG>, 21414), (<NAME>, 10602), (<CITATION>,..."
test/pool,99.8,154.77,"[(<ORG>, 168211), (<NAME>, 83008), (<CITATION>..."


citation marker used below: <CITATION>


,DATE OF JUDGMENT header %,CITATOR INFO %
split/role,,
train/queries,42.6,18.4
train/pool,43.0,22.8
dev/queries,47.5,22.9
dev/pool,54.6,31.1
test/queries,42.6,21.9
test/pool,47.5,25.7


,pool ids shared,pool texts shared,queries shared,train queries in dev pool,dev queries in train pool,train queries in test pool,test queries in train pool,dev queries in test pool,test queries in dev pool
pair,,,,,,,,,
train vs dev,436,437,0,124,90,0,0,0,0
train vs test,740,742,0,0,0,269,170,0,0
dev vs test,220,222,0,0,0,0,0,43,40


## 4. The labels: relevant cases per query, best possible score, popular candidates

Because we must always submit 7 ids, a query with 2 relevant cases can give at most 2 hits (precision <= 2/7),
and a query with 15 relevant cases at most 7 hits (recall <= 7/15). `oracle_micro_f1` = the score of a perfect
ranker under this rule = our ceiling. `random` = expected score of 7 random candidates (the floor).


In [5]:
def label_stats(split):
    Q, C = DATA[split]["queries"], DATA[split]["pool"]
    gold = {q["id"]: list(dict.fromkeys(q["rel"])) for q in Q}
    n = np.array([len(g) for g in gold.values()])
    N = len(C)
    rand_hits = sum(K * len(g) / N for g in gold.values())
    rp, rr = rand_hits / (K * len(gold)), rand_hits / n.sum()
    pop = collections.Counter(c for g in gold.values() for c in g)
    counts = np.array(sorted(pop.values(), reverse=True))
    top1 = max(1, int(0.01 * N))
    qids = {q["id"] for q in Q}
    bins = {"1": (n == 1).mean(), "2-3": ((n >= 2) & (n <= 3)).mean(), "4-7": ((n >= 4) & (n <= 7)).mean(),
            "8-15": ((n >= 8) & (n <= 15)).mean(), ">15": (n > 15).mean()}
    return {"queries": len(gold), "pool": N, "gold pairs": int(n.sum()), "gold/query": pct_row(n),
            "share of queries by #gold %": {k: round(100 * float(v), 1) for k, v in bins.items()},
            "oracle": {k: round(v, 4) for k, v in pu.oracle_micro_f1(gold).items()},
            "random": {"micro_f1": round(float(2 * rp * rr / (rp + rr)), 4), "micro_p": round(float(rp), 4), "micro_r": round(float(rr), 4)},
            "candidates ever relevant %": round(100 * len(pop) / N, 1),
            "times a relevant candidate is relevant": pct_row(counts),
            "gold pairs on top-1% candidates %": round(100 * counts[:top1].sum() / n.sum(), 1),
            "gold ids that are also queries": sum(c in qids for c in pop)}

LABELS = cached("labels.json", lambda: {s: label_stats(s) for s in ("train", "dev")})
for s, st in LABELS.items():
    print(f"===== {s}")
    for k, v in st.items():
        print(f"  {k:40s} {v}")

[cache] labels.json
===== train
  queries                                  827
  pool                                     4320
  gold pairs                               5308
  gold/query                               {'n': 827, 'mean': 6.4, 'p10': 1, 'p50': 4, 'p90': 14, 'p99': 31, 'max': 52}
  share of queries by #gold %              {'1': 14.6, '2-3': 28.3, '4-7': 25.9, '8-15': 22.1, '>15': 7.9}
  oracle                                   {'micro_f1': 0.6386, 'micro_p': 0.612, 'micro_r': 0.6675}
  random                                   {'micro_f1': 0.0016, 'micro_p': 0.0015, 'micro_r': 0.0016}
  candidates ever relevant %               95.1
  times a relevant candidate is relevant   {'n': 4107, 'mean': 1.3, 'p10': 1, 'p50': 1, 'p90': 2, 'p99': 5, 'max': 14}
  gold pairs on top-1% candidates %        5.0
  gold ids that are also queries           616
===== dev
  queries                                  118
  pool                                     1023
  gold pairs                 

**Does popularity transfer?** If the same candidates appear in the train and dev pools, a candidate that is often
relevant in train may also be relevant in dev (a "prior" that needs no text at all). Only meaningful if the pools
share ids (table in section 3).


In [6]:
def popularity_baseline():
    tr_pop = collections.Counter(c for q in DATA["train"]["queries"] for c in set(q["rel"]))
    out = {}
    for split in ("dev",):
        C = DATA[split]["pool"]
        ids = [d["id"] for d in C]
        shared = sum(i in tr_pop for i in ids)
        s = np.array([tr_pop.get(i, 0) for i in ids], dtype=float)
        pred = {q["id"]: pu.top_k(s, ids, exclude={q["id"]}) for q in DATA[split]["queries"]}
        gold = {q["id"]: q["rel"] for q in DATA[split]["queries"]}
        out[split] = {"pool ids seen as train gold": shared, **{k: round(v, 4) for k, v in pu.micro_prf_at_k(pred, gold).items() if k.startswith("micro")}}
    return out
POP = cached("popularity_baseline.json", popularity_baseline)
print(POP)

[cache] popularity_baseline.json
{'dev': {'pool ids seen as train gold': 396, 'micro_f1': 0.0122, 'micro_p': 0.0133, 'micro_r': 0.0113}}


## 5. Lengths: words, sentences, bpe32k tokens

My Phase-1 models read at most **1536 tokens** at once. A retrieval model must turn a whole case into a vector
(or compare two cases), so we need to know how much of a case fits, and how many chunks a whole case needs.
Tokens are counted on a sample after the Phase-1 cleaning (`clean_pcr` = the same cleaning as Phases 2-6).


In [7]:
def length_stats():
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(str(TOK_PATH))
    rng = np.random.default_rng(0)
    out = {}
    for split in DATA:
        for role in ("queries", "pool"):
            docs = DATA[split][role]
            words = [len(pu.doc_text(d).split()) for d in docs]
            sents = [len(d["sents"]) for d in docs]
            wps = [len(s.split()) for d in docs[:2000] for s in d["sents"]]
            idx = rng.choice(len(docs), size=min(N_TOK_SAMPLE, len(docs)), replace=False)
            texts = [pu.clean_pcr(pu.doc_text(docs[i])) for i in tqdm(idx, desc=f"clean {split}/{role}", leave=False)]
            ntok = np.array([len(e.ids) for e in tok.encode_batch(texts)])
            nw = np.array([max(len(t.split()), 1) for t in texts])
            ratio = float(ntok.sum() / nw.sum())
            total_tok = ratio * sum(words)
            out[f"{split}/{role}"] = {
                "docs": len(docs), "words": pct_row(words), "sentences": pct_row(sents), "words/sentence": pct_row(wps),
                "tokens (sample)": pct_row(ntok), "tokens/word": round(ratio, 3),
                "fit in tokens %": {str(L): round(100 * float((ntok <= L).mean()), 1) for L in (512, 1024, 1536, 3072, 6144, 12288)},
                "chunks of 1536 for the whole case (sample)": pct_row(np.ceil(ntok / 1536)),
                "total tokens (est., M)": round(total_tok / 1e6, 1),
                "one pass over all docs (est., min)": round(total_tok / FWD_TOK_PER_S / 60, 1)}
    return out

LENGTHS = cached("lengths.json", length_stats)
rows = []
for k, v in LENGTHS.items():
    rows.append({"split/role": k, "docs": v["docs"], "words p50": v["words"]["p50"], "words p90": v["words"]["p90"],
                 "words max": v["words"]["max"], "sentences p50": v["sentences"]["p50"], "tokens p50": v["tokens (sample)"]["p50"],
                 "tok/word": v["tokens/word"], "fit 1536 %": v["fit in tokens %"]["1536"],
                 "fit 6144 %": v["fit in tokens %"]["6144"],
                 "chunks p50": v["chunks of 1536 for the whole case (sample)"]["p50"],
                 "total tok (M)": v["total tokens (est., M)"], "encode all (min)": v["one pass over all docs (est., min)"]})
display(pd.DataFrame(rows).set_index("split/role"))

[cache] lengths.json


,docs,words p50,words p90,words max,sentences p50,tokens p50,tok/word,fit 1536 %,fit 6144 %,chunks p50,total tok (M),encode all (min)
split/role,,,,,,,,,,,,
train/queries,827,4508,12602,119513,37,5648,1.253,6.4,54.3,4,7.1,2.4
train/pool,4320,3632,12422,410597,30,4541,1.253,14.0,63.4,3,34.1,11.4
dev/queries,118,4619,15333,106169,38,5619,1.250,2.5,53.4,4,1.2,0.4
dev/pool,1023,4036,16598,410597,35,5065,1.248,13.3,58.9,4,11.0,3.7
test/queries,237,4653,14064,91838,40,5885,1.255,10.5,52.7,4,2.0,0.7
test/pool,1727,3947,13490,410597,34,4877,1.254,12.5,59.0,4,15.1,5.0


**Where are the citations inside a query?** If the query hides its citations as `<CITATION>`, the sentences
around these markers are exactly where the query "talks about" its prior cases. Position = sentence index /
number of sentences (0 = start of the case, 1 = end).


In [8]:
def citation_positions():
    if CIT is None:
        return {"note": "no placeholder found"}
    out = {}
    for split in DATA:
        pos, share, n = [], [], []
        for q in DATA[split]["queries"]:
            idx = [i for i, s in enumerate(q["sents"]) if CIT in s]
            n.append(len(idx))
            share.append(len(idx) / max(len(q["sents"]), 1))
            pos += [i / max(len(q["sents"]) - 1, 1) for i in idx]
        h = np.histogram(pos, bins=5, range=(0, 1))[0] if pos else np.zeros(5)
        out[split] = {"citation sentences per query": pct_row(n), "queries without any %": round(100 * float(np.mean(np.array(n) == 0)), 1),
                      "share of query sentences with a citation %": round(100 * float(np.mean(share)), 1),
                      "position histogram (0-20%,...,80-100%) %": [round(100 * float(x) / max(float(h.sum()), 1), 1) for x in h]}
    return out
CITPOS = cached("citation_positions.json", citation_positions)
for k, v in CITPOS.items():
    print(k, v)

[cache] citation_positions.json
train {'citation sentences per query': {'n': 827, 'mean': 5.7, 'p10': 1, 'p50': 4, 'p90': 13, 'p99': 32, 'max': 80}, 'queries without any %': 0.1, 'share of query sentences with a citation %': 13.0, 'position histogram (0-20%,...,80-100%) %': [14.3, 17.2, 19.9, 25.5, 23.0]}
dev {'citation sentences per query': {'n': 118, 'mean': 7.4, 'p10': 1, 'p50': 4, 'p90': 15, 'p99': 61, 'max': 76}, 'queries without any %': 0.0, 'share of query sentences with a citation %': 13.2, 'position histogram (0-20%,...,80-100%) %': [14.6, 18.8, 21.9, 24.2, 20.5]}
test {'citation sentences per query': {'n': 237, 'mean': 6.3, 'p10': 1, 'p50': 4, 'p90': 14, 'p99': 28, 'max': 36}, 'queries without any %': 0.0, 'share of query sentences with a citation %': 13.8, 'position histogram (0-20%,...,80-100%) %': [14.4, 17.6, 22.7, 24.5, 20.9]}


## 6. Years: are the relevant cases really _prior_ (older than the query)?

We estimate a case's year from its header `DATE OF JUDGMENT: dd/mm/yyyy` when it has one (about half of the cases),
otherwise as the **latest year written in it** (a judgment mentions older events and cites older cases, so its own
date is usually the newest year). The `CITATOR INFO` header block lists _later_ cases that cite this one, so it is
removed first. The estimate is noisy, so the table shows a **margin**: remove candidates more than `m` years
newer than the query -> how much of the pool goes away vs how many relevant cases we would lose. If almost every relevant candidate is not newer
than its query, a **year filter** (push candidates newer than the query to the bottom) is a free, explainable
improvement for every model.


In [9]:
MARGINS = (0, 1, 2, 3, 5, 10)

def compute_years():
    for split in DATA:
        YEARS[split] = {"q": {d["id"]: pu.doc_year(pu.doc_text(d), return_source=True) for d in DATA[split]["queries"]},
                        "c": {d["id"]: pu.doc_year(pu.doc_text(d), return_source=True) for d in tqdm(DATA[split]["pool"], desc=f"years {split}", leave=False)}}
        for k in ("q", "c"):
            YEARS[split][k + "_src"] = {i: v[1] for i, v in YEARS[split][k].items()}
            YEARS[split][k] = {i: v[0] for i, v in YEARS[split][k].items()}

def year_stats():
    out = {}
    for split in DATA:
        Q = DATA[split]["queries"]
        qy, cy = YEARS[split]["q"], YEARS[split]["c"]
        rec = {"queries with a year %": round(100 * np.mean([v is not None for v in qy.values()]), 1),
               "candidates with a year %": round(100 * np.mean([v is not None for v in cy.values()]), 1),
               "candidate year from header %": round(100 * np.mean([v == "header" for v in YEARS[split]["c_src"].values()]), 1),
               "query year": pct_row([v for v in qy.values() if v]), "candidate year": pct_row([v for v in cy.values() if v])}
        cyear = np.array([v or 0 for v in cy.values()])
        table = {}
        for m in MARGINS:
            row = {"pool removed %": round(100 * float(np.mean([(cyear > qy[q["id"]] + m).mean() for q in Q if qy[q["id"]]])), 1)}
            if split != "test":
                pairs = [(qy[q["id"]], cy[c]) for q in Q for c in q["rel"] if qy[q["id"]] and cy.get(c)]
                row["gold removed %"] = round(100 * float(np.mean([c > q + m for q, c in pairs])), 2)
            table[f"cand year > query year + {m}"] = row
        rec["filter table"] = table
        if split != "test":
            d = np.array([qy[q["id"]] - cy[c] for q in Q for c in q["rel"] if qy[q["id"]] and cy.get(c)])
            rec["gold pairs with both years"] = len(d)
            rec["years between query and gold (gold older)"] = pct_row(d[d >= 0])
        out[split] = rec
    return out

YEARS = {}
compute_years()            # cheap (regex), always recomputed: section 7 needs the per-case years
YEAR_STATS = cached("years.json", year_stats)
for k, v in YEAR_STATS.items():
    print(f"===== {k}")
    for a, b in v.items():
        if a != "filter table":
            print(f"  {a:45s} {b}")
    display(pd.DataFrame(v["filter table"]).T)

[cache] years.json
===== train
  queries with a year %                         99.8
  candidates with a year %                      98.0
  candidate year from header %                  42.9
  query year                                    {'n': 825, 'mean': 1970.0, 'p10': 1922, 'p50': 1975, 'p90': 2004, 'p99': 2014, 'max': 2025}
  candidate year                                {'n': 4233, 'mean': 1964.3, 'p10': 1914, 'p50': 1968, 'p90': 2002, 'p99': 2014, 'max': 2025}
  gold pairs with both years                    5218
  years between query and gold (gold older)     {'n': 4812, 'mean': 17.4, 'p10': 2, 'p50': 13, 'p90': 40, 'p99': 76, 'max': 138}


,pool removed %,gold removed %
cand year > query year + 0,43.4,7.78
cand year > query year + 1,42.4,7.17
cand year > query year + 2,41.4,6.73
cand year > query year + 3,40.5,6.40
cand year > query year + 5,38.5,6.02
cand year > query year + 10,34.0,5.23


===== dev
  queries with a year %                         100.0
  candidates with a year %                      99.2
  candidate year from header %                  54.6
  query year                                    {'n': 118, 'mean': 1974.0, 'p10': 1933, 'p50': 1977, 'p90': 2002, 'p99': 2012, 'max': 2014}
  candidate year                                {'n': 1015, 'mean': 1968.8, 'p10': 1928, 'p50': 1971, 'p90': 2001, 'p99': 2014, 'max': 2022}
  gold pairs with both years                    965
  years between query and gold (gold older)     {'n': 894, 'mean': 18.4, 'p10': 2, 'p50': 13, 'p90': 45, 'p99': 86, 'max': 120}


,pool removed %,gold removed %
cand year > query year + 0,43.0,7.36
cand year > query year + 1,41.9,6.94
cand year > query year + 2,40.7,6.74
cand year > query year + 3,39.7,6.22
cand year > query year + 5,37.6,5.91
cand year > query year + 10,32.5,4.66


===== test
  queries with a year %                         98.7
  candidates with a year %                      98.0
  candidate year from header %                  47.5
  query year                                    {'n': 234, 'mean': 1971.6, 'p10': 1925, 'p50': 1976, 'p90': 2003, 'p99': 2011, 'max': 2021}
  candidate year                                {'n': 1693, 'mean': 1966.9, 'p10': 1919, 'p50': 1972, 'p90': 2001, 'p99': 2014, 'max': 2025}


,pool removed %
cand year > query year + 0,44.0
cand year > query year + 1,43.0
cand year > query year + 2,41.9
cand year > query year + 3,40.9
cand year > query year + 5,38.8
cand year > query year + 10,34.0


## 7. BM25 baselines (dev)

**BM25** scores a candidate by the query words it contains: rare words (high _idf_) count more, a word repeated
many times saturates (parameter `k1`), long candidates are penalised a bit (parameter `b`). No learning, no
pretrained model - it is the standard lexical baseline in legal retrieval (and the usual first stage before a
neural re-ranker). Every row below changes **one** thing:

- query words counted once vs. as often as they occur (`qtf`)
- `k1` / `b` grid
- **which part of the case** is used: first N words of query and candidates (= what a model with a limited window sees),
  or only the query sentences around the citation markers
- **year filter** (section 6)

`R@K` = share of all relevant pairs that are in the top K -> how many candidates a neural re-ranker must look at.


In [10]:
INDEX, QTOK = {}, {}

def words_first(d, n):
    t = pu.doc_text(d)
    return t if n is None else " ".join(t.split()[:n])

def cit_text(d, window=1):
    # the query sentences that contain the citation marker, plus `window` sentences on each side
    if CIT is None:
        return pu.doc_text(d)
    keep = set()
    for i, s in enumerate(d["sents"]):
        if CIT in s:
            keep.update(range(max(0, i - window), min(len(d["sents"]), i + window + 1)))
    return " ".join(d["sents"][i] for i in sorted(keep)) or pu.doc_text(d)

def get_index(split, c_words=None):
    key = (split, c_words)
    if key not in INDEX:
        docs = DATA[split]["pool"]
        INDEX[key] = pu.BM25((pu.bm25_tokens(words_first(d, c_words)) for d in docs), desc=f"index {split} first {c_words}")
    return INDEX[key]

def get_qtok(split, mode="full", q_words=None):
    # dev query words are kept in memory (re-used by ~40 settings); train / test are produced one query at a time
    f = (lambda d: cit_text(d)) if mode == "cit" else (lambda d: words_first(d, q_words))
    if split != "dev":
        return (pu.bm25_tokens(f(d)) for d in DATA[split]["queries"])
    key = (split, mode, q_words)
    if key not in QTOK:
        QTOK[key] = [pu.bm25_tokens(f(d)) for d in DATA[split]["queries"]]
    return QTOK[key]

def run_bm25(split, k1=1.2, b=0.75, qtf=False, mode="full", q_words=None, c_words=None, year_margin=None, return_scores=False):
    bm = get_index(split, c_words)
    bm.set_params(k1, b)
    Q = DATA[split]["queries"]
    S = bm.score_many(get_qtok(split, mode, q_words), query_tf=qtf, desc=f"score {split}", total=len(Q))
    ids = [d["id"] for d in DATA[split]["pool"]]
    if year_margin is not None:
        cy = np.array([YEARS[split]["c"][i] or 0 for i in ids])
        for r, q in enumerate(Q):
            qy = YEARS[split]["q"][q["id"]]
            if qy:
                S[r, cy > qy + year_margin] -= 1e6     # clearly newer than the query -> not a prior case -> bottom
    if return_scores:
        return S
    pred = {q["id"]: pu.top_k(S[r], ids, exclude={q["id"]}) for r, q in enumerate(Q)}
    gold = {q["id"]: q["rel"] for q in Q}
    ranks = [x for r, q in enumerate(Q) for x in pu.gold_ranks(S[r], ids, q["rel"], exclude={q["id"]})]
    m = pu.micro_prf_at_k(pred, gold)
    rk = np.array(ranks, dtype=float)
    return {"micro_f1": round(m["micro_f1"], 4), "micro_p": round(m["micro_p"], 4), "micro_r": round(m["micro_r"], 4),
            **pu.recall_at_k(ranks), "median gold rank": float(np.median(rk[np.isfinite(rk)])) if np.isfinite(rk).any() else None}

def bm25_dev():
    # Greedy, one change at a time: each step keeps the change only if dev Micro-F1@7 goes up.
    res = {}
    def add(name, **kw):
        t0 = time.time()
        res[name] = {**run_bm25("dev", **kw), "settings": dict(kw), "sec": round(time.time() - t0, 1)}
        print(f"{name:52s} F1@7 {res[name]['micro_f1']:.4f}  R@100 {res[name]['R@100']:.3f}  ({res[name]['sec']} s)")
        return res[name]["micro_f1"]
    cur = {"k1": 1.2, "b": 0.75}
    best_f1 = add("A default, full cases (k1 1.2, b 0.75)", **cur)
    # step B: how much of each case to read (first N words of query AND candidates)
    for n in (512, 1024, 2048, 4096, 8192):
        f = add(f"B first {n} words (query and candidates)", q_words=n, c_words=n, **{k: v for k, v in cur.items() if k in ("k1", "b")})
        if f > best_f1:
            best_f1, cur = f, {**cur, "q_words": n, "c_words": n}
    L = cur.get("c_words")
    add(f"B full query, candidates first {L} words", c_words=L, k1=cur["k1"], b=cur["b"])
    add(f"B query first {L} words, full candidates", q_words=L, k1=cur["k1"], b=cur["b"])
    for name in (f"B full query, candidates first {L} words", f"B query first {L} words, full candidates"):
        if res[name]["micro_f1"] > best_f1:
            best_f1, cur = res[name]["micro_f1"], {**cur, **res[name]["settings"]}
            cur = {k: v for k, v in cur.items() if v is not None}
    # step C: k1 x b grid at that length
    base = {k: v for k, v in cur.items() if k not in ("k1", "b")}
    for k1 in (0.9, 1.2, 2.0, 3.0, 5.0, 8.0):
        for b in (0.3, 0.5, 0.75, 0.9, 1.0):
            f = add(f"C grid k1 {k1} b {b}", k1=k1, b=b, **base)
            if f > best_f1:
                best_f1, cur = f, {**base, "k1": k1, "b": b}
    # step D: query words with counts, citation sentences, year filter
    f = add("D + query words counted (qtf)", qtf=True, **cur)
    if f > best_f1:
        best_f1, cur = f, {**cur, "qtf": True}
    if CIT is not None:
        f = add("D + query = citation sentences +-1", mode="cit", **{k: v for k, v in cur.items() if k != "q_words"})
        if f > best_f1:
            best_f1, cur = f, {**{k: v for k, v in cur.items() if k != "q_words"}, "mode": "cit"}
    bm = None
    for m in (0, 2, 5, 10):
        f = add(f"D + year filter (margin {m})", year_margin=m, **cur)
        if f > best_f1:
            best_f1, bm = f, m
    if bm is not None:
        cur = {**cur, "year_margin": bm}
    res["_best"] = {"settings": cur, "micro_f1": best_f1}
    print("BEST:", cur, round(best_f1, 4))
    return res

BM25_DEV = cached("bm25_dev.json", bm25_dev)
tab = pd.DataFrame({k: v for k, v in BM25_DEV.items() if not k.startswith("_")}).T.drop(columns=["settings"])
display(tab[~tab.index.str.startswith("C grid")])
grid = tab[tab.index.str.startswith("C grid")]["micro_f1"].astype(float)
g = pd.DataFrame([(float(i.split()[3]), float(i.split()[5]), v) for i, v in grid.items()], columns=["k1", "b", "F1"])
print("grid (rows k1, columns b), Micro-F1@7:")
display(g.pivot(index="k1", columns="b", values="F1"))
BEST = BM25_DEV["_best"]["settings"]
print("best BM25 setting on dev:", BEST, "F1@7", round(BM25_DEV["_best"]["micro_f1"], 4))

[cache] bm25_dev.json


,micro_f1,micro_p,micro_r,R@7,R@20,R@50,R@100,R@200,R@500,median gold rank,sec
"A default, full cases (k1 1.2, b 0.75)",0.1067,0.1162,0.0987,0.0987,0.1531,0.2364,0.3227,0.519,0.7811,191.0,3.7
B first 512 words (query and candidates),0.2779,0.3027,0.2569,0.2569,0.4049,0.5242,0.6269,0.7369,0.8839,42.0,0.9
B first 1024 words (query and candidates),0.3124,0.3402,0.2888,0.2888,0.4481,0.5766,0.6578,0.7431,0.8767,29.0,1.1
B first 2048 words (query and candidates),0.3268,0.3559,0.3022,0.3022,0.4543,0.5766,0.6557,0.7575,0.8808,29.0,1.5
B first 4096 words (query and candidates),0.3235,0.3523,0.2991,0.2991,0.4183,0.5262,0.6002,0.7009,0.8407,40.0,2.0
B first 8192 words (query and candidates),0.2646,0.2881,0.2446,0.2446,0.333,0.4265,0.5077,0.6095,0.8088,95.0,2.8
"B full query, candidates first 2048 words",0.3535,0.385,0.3268,0.3268,0.4584,0.5612,0.6465,0.739,0.8654,30.0,0.5
"B query first 2048 words, full candidates",0.1334,0.1453,0.1233,0.1233,0.1973,0.3042,0.4121,0.5714,0.815,153.0,0.3
D + query words counted (qtf),0.3313,0.3608,0.3063,0.3063,0.4666,0.5992,0.7112,0.8078,0.9198,24.0,0.3
D + query = citation sentences +-1,0.4036,0.4395,0.3731,0.3731,0.5601,0.6701,0.7235,0.7955,0.9034,14.0,0.3


grid (rows k1, columns b), Micro-F1@7:


b,0.30,0.50,0.75,0.90,1.00
k1,,,,,
0.9,0.3213,0.3224,0.3246,0.3268,0.3280
1.2,0.3224,0.3257,0.3268,0.3291,0.3291
2.0,0.3213,0.3246,0.3324,0.3369,0.3413
3.0,0.3235,0.3280,0.3391,0.3435,0.3480
5.0,0.3324,0.3346,0.3457,0.3480,0.3513
8.0,0.3313,0.3357,0.3491,0.3580,0.3613


best BM25 setting on dev: {'c_words': 2048, 'k1': 8.0, 'b': 1.0, 'mode': 'cit', 'year_margin': 0} F1@7 0.4247


**Train split** (bigger pool = harder): the default and the best dev setting, mainly to see the ranks of the
relevant candidates - BM25's high-ranked _wrong_ candidates are the natural **hard negatives** for training a
neural ranker (cases that look similar in words but are not cited).


In [11]:
def bm25_train():
    return {"default (full cases)": run_bm25("train"),
            "best dev setting without year filter": run_bm25("train", **{k: v for k, v in BEST.items() if k != "year_margin"}),
            "best dev setting": run_bm25("train", **BEST)}
BM25_TRAIN = cached("bm25_train.json", bm25_train)
display(pd.DataFrame(BM25_TRAIN).T)

[cache] bm25_train.json


,micro_f1,micro_p,micro_r,R@7,R@20,R@50,R@100,R@200,R@500,median gold rank
default (full cases),0.1065,0.1021,0.1113,0.1113,0.1669,0.2281,0.2941,0.3877,0.5390,413.5
best dev setting without year filter,0.3181,0.3049,0.3325,0.3325,0.4994,0.6258,0.7042,0.7707,0.8493,21.0
best dev setting,0.3455,0.3311,0.3612,0.3612,0.5175,0.6326,0.7097,0.7658,0.8370,18.0


**Which design choice is best is decided by the numbers above; one failure example** (best BM25 setting on dev):
the top-7 for one query with hits marked, and where its relevant cases ended up.


In [12]:
S_dev = run_bm25("dev", return_scores=True, **BEST)
ids = [d["id"] for d in DATA["dev"]["pool"]]
by_id = {d["id"]: d for d in DATA["dev"]["pool"]}
Qd = DATA["dev"]["queries"]
# a typical query: the one with the median number of hits among queries with >= 3 gold
cand = [(r, q) for r, q in enumerate(Qd) if len(q["rel"]) >= 3] or list(enumerate(Qd))
hits = [len(set(pu.top_k(S_dev[r], ids, exclude={q["id"]})) & set(q["rel"])) for r, q in cand]
r, q = cand[int(np.argsort(hits)[len(hits) // 2])]
top = pu.top_k(S_dev[r], ids, exclude={q["id"]})
print(f"query {q['id']} ({len(q['sents'])} sentences, year {YEARS['dev']['q'][q['id']]}), gold {len(q['rel'])}:")
print("   |", pu.doc_text(q)[:400])
for i, c in enumerate(top, 1):
    print(f"{i}. {'HIT ' if c in q['rel'] else '    '}{c} (year {YEARS['dev']['c'][c]}) | {pu.doc_text(by_id[c])[:160]}")
print("ranks of the gold cases:", sorted(pu.gold_ranks(S_dev[r], ids, q["rel"], exclude={q["id"]})))

query 0000783511 (40 sentences, year 1974), gold 8:
   | <CITATION> .  1. <ORG> has filed this writ petition under Article 226 of the Constitution praying that <ORG> may be restrained by issuing a writ of prohibition from levying and collecting excise duty on their product, namely, woollen industrial felt under the Central Excises and Salt Act , 1944 (hereinafter referred to as 'the Act'). It was further prayed that the amount of Rs. 7,52,320.55 paise c
1. HIT 0001650570 (year 1964) | JUDGMENT <NAME>, J.  1. The short question that arises on this reference is whether borders woven on handloom out of pure silk, art silk and jari thread are han
2. HIT 0000453569 (year 1948) | JUDGMENT <NAME>, J.  1. The appellants carry on business of buying paddy and after getting it husked either in their own mills or in other mills sell the rice t
3. HIT 0001822153 (year 1967) | <NAME>, J.  1. These cases raise the question of the scope of entry 47 in Schedule I to the Madras General Sales Tax Act, 19

## 8. Reference submission (BM25, best dev setting) - **NOT for upload**

It does not use a Phase-1 model, so it would not count. It only tests the writer and the checker on the real test
pool (exactly 7 unique ids per query, all from `test_candidates.json`, never the query itself), and gives the
lexical reference point for the report.


In [13]:
REF_DIR = OUT / "bm25_reference"
zip_path = REF_DIR / "submission.zip"
Qt, Ct = DATA["test"]["queries"], DATA["test"]["pool"]
if not zip_path.exists():
    S_test = run_bm25("test", return_scores=True, **BEST)
    tid = [d["id"] for d in Ct]
    pu.write_submission(REF_DIR, {q["id"]: pu.top_k(S_test[r], tid, exclude={q["id"]}) for r, q in enumerate(Qt)})
    print("setting used:", BEST)
print(pu.check_submission(zip_path, [q["id"] for q in Qt], [d["id"] for d in Ct]))

{'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 983}


## 9. Summary (saved to `data_summary.json`)


In [14]:
best_name = max((k for k in BM25_DEV if not k.startswith("_")), key=lambda k: BM25_DEV[k]["micro_f1"])
assert abs(BM25_DEV[best_name]["micro_f1"] - BM25_DEV["_best"]["micro_f1"]) < 1e-9
SUMMARY = {
    "format": {s: {r: {k: INFO[s][r][k] for k in ("file_type", "records", "keys", "id_type_in_file", "text_type", "rel_key")} for r in ("queries", "pool")} for s in INFO},
    "checks": CHECKS, "placeholder": CIT, "labels": LABELS, "popularity": POP,
    "lengths": {k: {"words_p50": v["words"]["p50"], "tokens_p50": v["tokens (sample)"]["p50"], "tok_per_word": v["tokens/word"],
                    "fit": v["fit in tokens %"], "total_tok_M": v["total tokens (est., M)"]} for k, v in LENGTHS.items()},
    "citations": CITPOS, "years": YEAR_STATS,
    "bm25_dev_best": {"name": best_name, **{k: v for k, v in BM25_DEV[best_name].items() if k != "settings"}},
    "bm25_dev_default": BM25_DEV["A default, full cases (k1 1.2, b 0.75)"], "bm25_best_settings": BEST, "bm25_train": BM25_TRAIN,
}
(OUT / "data_summary.json").write_text(json.dumps(SUMMARY, indent=1, default=str), encoding="utf-8")
print(json.dumps({k: SUMMARY[k] for k in ("placeholder", "bm25_dev_best", "bm25_dev_default")}, indent=1, default=str))
print("oracle dev:", LABELS["dev"]["oracle"], " random dev:", LABELS["dev"]["random"])

{
 "placeholder": "<CITATION>",
 "bm25_dev_best": {
  "name": "D + year filter (margin 0)",
  "micro_f1": 0.4247,
  "micro_p": 0.4625,
  "micro_r": 0.3926,
  "R@7": 0.3926,
  "R@20": 0.5612,
  "R@50": 0.6608,
  "R@100": 0.7307,
  "R@200": 0.7934,
  "R@500": 0.8911,
  "median gold rank": 13.0,
  "sec": 0.2
 },
 "bm25_dev_default": {
  "micro_f1": 0.1067,
  "micro_p": 0.1162,
  "micro_r": 0.0987,
  "R@7": 0.0987,
  "R@20": 0.1531,
  "R@50": 0.2364,
  "R@100": 0.3227,
  "R@200": 0.519,
  "R@500": 0.7811,
  "median gold rank": 191.0,
  "settings": {
   "k1": 1.2,
   "b": 0.75
  },
  "sec": 3.7
 }
}
oracle dev: {'micro_f1': 0.6181, 'micro_p': 0.6731, 'micro_r': 0.5714}  random dev: {'micro_f1': 0.0074, 'micro_p': 0.0081, 'micro_r': 0.0068}
